# Notebook 05 — Clinical Dosing Impact Mapping (CPIC)

**Author:** Nandan Kumar K N  
**Project:** AI-Driven Pharmacogenomics — Asian ethnic subgroups  
**Goal:** Map predicted metaboliser phenotypes to CPIC dosing recommendations. Identify which subgroup-drug combinations require dose adjustment. Produce Figure 9 (drug-population heatmap) — the translational output of the paper.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

ROOT     = Path(r"D:\GIT\asian-pgx-ml")
PROC_DIR = ROOT / 'data' / 'processed'
FIG_DIR  = ROOT / 'results' / 'figures'
TAB_DIR  = ROOT / 'results' / 'tables'

for d in [FIG_DIR, TAB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

POPS = ['GIH', 'ITU', 'BEB', 'CHB', 'CHS', 'JPT']
SAS  = ['GIH', 'ITU', 'BEB']
EAS  = ['CHB', 'CHS', 'JPT']

POP_LABELS = {
    'GIH': 'Gujarati\nIndian (GIH)',
    'ITU': 'Indian\nTelugu (ITU)',
    'BEB': 'Bengali\n(BEB)',
    'CHB': 'Han Chinese\n(CHB)',
    'CHS': 'S. Han\nChinese (CHS)',
    'JPT': 'Japanese\n(JPT)',
}

print("✓ Imports OK")


## 1. CPIC dosing recommendation framework


In [ ]:
# ── CPIC Level A drug-gene-phenotype dosing recommendations ──────────────
# Source: cpicpgx.org guidelines (2024 versions)
# For each gene × phenotype, what does CPIC recommend?

CPIC_RECOMMENDATIONS = {
    'CYP2C19': {
        'PM': {
            'clopidogrel':  ('Avoid', 'Use alternative antiplatelet (prasugrel/ticagrelor)'),
            'omeprazole':   ('Reduce dose', '50% dose reduction — increased drug exposure'),
            'escitalopram': ('Reduce dose', '50% dose reduction — increased plasma levels'),
            'voriconazole': ('Avoid', 'Unpredictable exposure — use alternative antifungal'),
        },
        'IM': {
            'clopidogrel':  ('Consider alternative', 'Reduced activation — consider prasugrel'),
            'omeprazole':   ('Normal', 'Standard dosing — monitor response'),
            'escitalopram': ('Normal', 'Standard dosing'),
            'voriconazole': ('Normal', 'Standard dosing with TDM'),
        },
        'NM': {
            'clopidogrel':  ('Normal', 'Standard dosing'),
            'omeprazole':   ('Normal', 'Standard dosing'),
            'escitalopram': ('Normal', 'Standard dosing'),
            'voriconazole': ('Normal', 'Standard dosing'),
        },
    },
    'CYP2D6': {
        'PM': {
            'codeine':      ('Contraindicated', 'No morphine conversion — inadequate analgesia'),
            'tamoxifen':    ('Avoid', 'Severely reduced endoxifen — use aromatase inhibitor'),
            'tramadol':     ('Avoid', 'Reduced efficacy — inadequate analgesia'),
            'amitriptyline':('Reduce dose', '50% dose reduction — increased plasma levels'),
        },
        'IM': {
            'codeine':      ('Use with caution', 'Reduced conversion — monitor analgesia'),
            'tamoxifen':    ('Consider alternative', 'Reduced endoxifen — monitor or switch'),
            'tramadol':     ('Normal', 'Standard dosing with monitoring'),
            'amitriptyline':('Normal', 'Standard dosing'),
        },
        'NM': {
            'codeine':      ('Normal', 'Standard dosing'),
            'tamoxifen':    ('Normal', 'Standard dosing'),
            'tramadol':     ('Normal', 'Standard dosing'),
            'amitriptyline':('Normal', 'Standard dosing'),
        },
    },
}

# ── Severity scoring ──────────────────────────────────────────────────────
SEVERITY = {
    'Contraindicated': 4,
    'Avoid':           3,
    'Reduce dose':     2,
    'Consider alternative': 2,
    'Use with caution': 1,
    'Normal':          0,
}

SEVERITY_COLORS = {
    4: '#922B21',   # Contraindicated — dark red
    3: '#E74C3C',   # Avoid — red
    2: '#E67E22',   # Reduce/consider — orange
    1: '#F1C40F',   # Caution — yellow
    0: '#27AE60',   # Normal — green
}

print("✓ CPIC framework loaded")
print(f"  Genes    : {list(CPIC_RECOMMENDATIONS.keys())}")
print(f"  CYP2C19 drugs: {list(CPIC_RECOMMENDATIONS['CYP2C19']['PM'].keys())}")
print(f"  CYP2D6 drugs : {list(CPIC_RECOMMENDATIONS['CYP2D6']['PM'].keys())}")


## 2. Load feature matrix and assign phenotypes


In [ ]:
fm = pd.read_csv(PROC_DIR / 'feature_matrix_full.csv', index_col=0)

# ── CYP2C19 phenotype (confirmed *2 position) ─────────────────────────────
star2_col = 'CYP2C19_10:94842865:C>T'
fm[star2_col] = pd.to_numeric(fm[star2_col], errors='coerce').fillna(0)

def assign_cyp2c19(row):
    d = row[star2_col]
    if d >= 2: return 'PM'
    if d == 1: return 'IM'
    return 'NM'

fm['CYP2C19_phenotype'] = fm.apply(assign_cyp2c19, axis=1)

# ── CYP2D6 phenotype (burden score) ──────────────────────────────────────
cyp2d6_cols = [c for c in fm.columns
               if c.startswith('CYP2D6_') and '_TPM_' not in c]
fm[cyp2d6_cols] = fm[cyp2d6_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
fm['CYP2D6_burden'] = fm[cyp2d6_cols].sum(axis=1)
b75 = fm['CYP2D6_burden'].quantile(0.75)
b90 = fm['CYP2D6_burden'].quantile(0.90)

def assign_cyp2d6(row):
    b = row['CYP2D6_burden']
    if b >= b90: return 'PM'
    if b >= b75: return 'IM'
    return 'NM'

fm['CYP2D6_phenotype'] = fm.apply(assign_cyp2d6, axis=1)

print("Phenotype distributions:")
print("\nCYP2C19:")
print(fm.groupby(['population', 'CYP2C19_phenotype']).size().unstack(fill_value=0))
print("\nCYP2D6:")
print(fm.groupby(['population', 'CYP2D6_phenotype']).size().unstack(fill_value=0))


## 3. Compute dose adjustment rates per population × drug


In [ ]:
records = []

for pop in POPS:
    pop_df = fm[fm['population'] == pop]
    n_total = len(pop_df)

    for gene, drugs in CPIC_RECOMMENDATIONS.items():
        pheno_col = f'{gene}_phenotype'
        pheno_counts = pop_df[pheno_col].value_counts()

        for drug in list(drugs['PM'].keys()):
            for pheno in ['PM', 'IM', 'NM']:
                n_pheno = pheno_counts.get(pheno, 0)
                recommendation, rationale = drugs[pheno][drug]
                severity = SEVERITY[recommendation]

                records.append({
                    'population':     pop,
                    'gene':           gene,
                    'drug':           drug,
                    'phenotype':      pheno,
                    'n_individuals':  n_pheno,
                    'n_total':        n_total,
                    'pct':            round(n_pheno / n_total * 100, 1),
                    'recommendation': recommendation,
                    'rationale':      rationale,
                    'severity':       severity,
                    'needs_action':   severity > 0,
                })

clinical_df = pd.DataFrame(records)

# ── Summary: % needing non-standard dosing per population × drug ──────────
action_df = clinical_df[clinical_df['needs_action']].groupby(
    ['population', 'drug']
).agg(
    pct_needing_action=('pct', 'sum'),
    max_severity=('severity', 'max')
).reset_index()

print("% of individuals needing non-standard dosing (PM + IM combined):")
pivot = action_df.pivot(index='drug', columns='population', values='pct_needing_action')
pivot = pivot.reindex(columns=POPS).fillna(0)
print(pivot.round(1).to_string())

clinical_df.to_csv(TAB_DIR / 'table5_cpic_clinical_mapping.csv', index=False)
print(f"\n✓ Saved → table5_cpic_clinical_mapping.csv")


## 4. Figure 9 — Drug × Population dosing heatmap


In [ ]:
# ── Build severity heatmap: worst-case severity per pop × drug ────────────
severity_pivot = action_df.pivot(
    index='drug', columns='population', values='max_severity'
).reindex(columns=POPS).fillna(0)

pct_pivot = pivot.copy()

# Drug display order: most clinically impactful first
drug_order = ['clopidogrel', 'voriconazole', 'tamoxifen', 'codeine',
              'omeprazole', 'escitalopram', 'tramadol', 'amitriptyline']
severity_pivot = severity_pivot.reindex(drug_order)
pct_pivot      = pct_pivot.reindex(drug_order)

# Gene label per drug
drug_gene = {
    'clopidogrel': 'CYP2C19', 'voriconazole': 'CYP2C19',
    'omeprazole': 'CYP2C19', 'escitalopram': 'CYP2C19',
    'codeine': 'CYP2D6', 'tamoxifen': 'CYP2D6',
    'tramadol': 'CYP2D6', 'amitriptyline': 'CYP2D6',
}

fig, ax = plt.subplots(figsize=(13, 7))

# Draw cells
for i, drug in enumerate(drug_order):
    for j, pop in enumerate(POPS):
        sev  = severity_pivot.loc[drug, pop] if pop in severity_pivot.columns else 0
        pct  = pct_pivot.loc[drug, pop] if pop in pct_pivot.columns else 0
        color = SEVERITY_COLORS.get(int(sev), '#27AE60')

        rect = mpatches.FancyBboxPatch(
            (j - 0.45, i - 0.45), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor='white', linewidth=1.5
        )
        ax.add_patch(rect)

        # Text: percentage
        text_color = 'white' if sev >= 2 else '#1a1a1a'
        ax.text(j, i, f'{pct:.0f}%', ha='center', va='center',
                fontsize=9, fontweight='bold', color=text_color)

# Axes
ax.set_xlim(-0.55, len(POPS) - 0.45)
ax.set_ylim(-0.55, len(drug_order) - 0.45)
ax.set_xticks(range(len(POPS)))
ax.set_xticklabels([POP_LABELS[p] for p in POPS], fontsize=8.5, ha='center')
ax.set_yticks(range(len(drug_order)))
ax.set_yticklabels([d.capitalize() for d in drug_order], fontsize=10)
ax.invert_yaxis()

# SAS / EAS divider
ax.axvline(2.55, color='#2C3E50', linewidth=2.5, linestyle='--', alpha=0.7)
ax.text(1.25, -0.75, 'South Asian (SAS)', ha='center', fontsize=9,
        color='#1565C0', fontweight='bold',
        transform=ax.get_xaxis_transform())
ax.text(4.25, -0.75, 'East Asian (EAS)', ha='center', fontsize=9,
        color='#C62828', fontweight='bold',
        transform=ax.get_xaxis_transform())

# Gene labels on right
gene_divider = 3.5   # after escitalopram
ax.axhline(gene_divider, color='#555555', linewidth=1, linestyle=':', alpha=0.5)
ax.text(len(POPS) - 0.4, 1.5, 'CYP2C19', fontsize=9, color='#1A5276',
        fontweight='bold', va='center', ha='left',
        rotation=90, transform=ax.transData)
ax.text(len(POPS) - 0.4, 5.5, 'CYP2D6', fontsize=9, color='#117A65',
        fontweight='bold', va='center', ha='left',
        rotation=90, transform=ax.transData)

# Legend
legend_elements = [
    mpatches.Patch(facecolor=SEVERITY_COLORS[4], label='Contraindicated'),
    mpatches.Patch(facecolor=SEVERITY_COLORS[3], label='Avoid — use alternative'),
    mpatches.Patch(facecolor=SEVERITY_COLORS[2], label='Reduce dose / consider alternative'),
    mpatches.Patch(facecolor=SEVERITY_COLORS[1], label='Use with caution'),
    mpatches.Patch(facecolor=SEVERITY_COLORS[0], label='Standard dosing (normal)'),
]
ax.legend(handles=legend_elements, loc='lower right',
          bbox_to_anchor=(1.0, -0.18), ncol=3,
          fontsize=8, title='CPIC recommendation', title_fontsize=8,
          framealpha=0.9)

ax.set_title('Figure 9: CPIC Clinical Dosing Impact Map
'
              '% of individuals requiring non-standard dosing by population and drug',
              fontsize=12, fontweight='bold', pad=15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.tick_params(left=False, bottom=False)

plt.tight_layout()
plt.savefig(FIG_DIR / 'figure9_cpic_dosing_heatmap.png', dpi=300,
            bbox_inches='tight', facecolor='white')
print("✓ Figure 9 saved")
plt.show()


## 5. High-priority clinical alerts


In [ ]:
# ── Flag subgroup-drug combinations where >20% need non-standard dosing ──
print("HIGH-PRIORITY CLINICAL ALERTS (>20% requiring non-standard dosing)")
print("="*65)

alerts = action_df[action_df['pct_needing_action'] > 20].sort_values(
    ['max_severity', 'pct_needing_action'], ascending=[False, False]
)

for _, row in alerts.iterrows():
    gene = drug_gene.get(row['drug'], '')
    sev_label = {4:'CONTRAINDICATED', 3:'AVOID', 2:'DOSE ADJUSTMENT', 1:'CAUTION'}.get(
        int(row['max_severity']), 'MONITOR')
    print(f"  [{sev_label}] {row['population']} × {row['drug'].capitalize()} "
          f"({gene}): {row['pct_needing_action']:.1f}% affected")

print(f"\nTotal high-priority alerts: {len(alerts)}")


## 6. Figure 10 — SAS vs EAS dosing comparison


In [ ]:
# ── Bar chart: mean % needing action, SAS vs EAS per drug ────────────────
sas_mean = action_df[action_df['population'].isin(SAS)].groupby('drug')['pct_needing_action'].mean()
eas_mean = action_df[action_df['population'].isin(EAS)].groupby('drug')['pct_needing_action'].mean()

comparison_df = pd.DataFrame({
    'SAS': sas_mean,
    'EAS': eas_mean,
}).reindex(drug_order).fillna(0)

fig, ax = plt.subplots(figsize=(11, 5))

x     = np.arange(len(drug_order))
width = 0.35

bars_sas = ax.bar(x - width/2, comparison_df['SAS'], width,
                   label='South Asian (SAS)', color='#1565C0',
                   edgecolor='white', linewidth=0.5, alpha=0.85)
bars_eas = ax.bar(x + width/2, comparison_df['EAS'], width,
                   label='East Asian (EAS)', color='#C62828',
                   edgecolor='white', linewidth=0.5, alpha=0.85)

# Value labels
for bar in bars_sas:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                f'{h:.0f}%', ha='center', va='bottom', fontsize=8)
for bar in bars_eas:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                f'{h:.0f}%', ha='center', va='bottom', fontsize=8)

# Gene divider
ax.axvline(3.5, color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax.text(1.75, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 50,
        'CYP2C19 drugs', ha='center', fontsize=8.5,
        color='#1A5276', fontweight='bold')
ax.text(5.5, ax.get_ylim()[1] * 0.95 if ax.get_ylim()[1] > 0 else 50,
        'CYP2D6 drugs', ha='center', fontsize=8.5,
        color='#117A65', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([d.capitalize() for d in drug_order], rotation=20, ha='right')
ax.set_ylabel('% individuals needing non-standard dosing', fontsize=10)
ax.set_title('Figure 10: SAS vs EAS Dosing Adjustment Rates
'
              'Mean across subgroups within each super-population',
              fontweight='bold', fontsize=11)
ax.legend(fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig(FIG_DIR / 'figure10_sas_eas_dosing_comparison.png', dpi=300,
            bbox_inches='tight', facecolor='white')
print("✓ Figure 10 saved")
plt.show()


In [ ]:
# ── Final paper summary table ─────────────────────────────────────────────
summary = action_df.groupby('drug').agg(
    SAS_mean_pct=('pct_needing_action',
                  lambda x: x[action_df.loc[x.index,'population'].isin(SAS)].mean()),
    EAS_mean_pct=('pct_needing_action',
                  lambda x: x[action_df.loc[x.index,'population'].isin(EAS)].mean()),
    max_severity=('max_severity', 'max')
).reset_index()

summary['gene'] = summary['drug'].map(drug_gene)
summary['severity_label'] = summary['max_severity'].map(
    {4:'Contraindicated', 3:'Avoid', 2:'Dose adjustment', 1:'Caution', 0:'Normal'})
summary = summary.reindex(
    summary.columns.tolist()
).sort_values('max_severity', ascending=False)

print("\nPaper Table 5 — Clinical Dosing Impact Summary:")
print("="*70)
print(summary[['drug','gene','SAS_mean_pct','EAS_mean_pct','severity_label']
              ].to_string(index=False, float_format='{:.1f}'.format))

summary.to_csv(TAB_DIR / 'table5_dosing_summary.csv', index=False)
print(f"\n✓ Saved → table5_dosing_summary.csv")


In [ ]:
print("="*60)
print("NOTEBOOK 05 COMPLETE")
print("="*60)

print(f"\nClinical findings for paper:")
print(f"  High-priority alerts (>20% needing action): {len(alerts)}")

top_alert = alerts.iloc[0] if len(alerts) > 0 else None
if top_alert is not None:
    print(f"  Highest impact: {top_alert['population']} × "
          f"{top_alert['drug'].capitalize()} "
          f"({top_alert['pct_needing_action']:.1f}% affected)")

print(f"\nFigures saved:")
print(f"  Figure 9  → figure9_cpic_dosing_heatmap.png")
print(f"  Figure 10 → figure10_sas_eas_dosing_comparison.png")

print(f"\nTables saved:")
print(f"  Table 5a  → table5_cpic_clinical_mapping.csv")
print(f"  Table 5b  → table5_dosing_summary.csv")

print(f"\n{'='*60}")
print(f"ALL 5 NOTEBOOKS COMPLETE")
print(f"{'='*60}")
print(f"\nYour analysis pipeline is done. Summary:")
print(f"  Notebook 01 → Allele frequency EDA, Figure 1-2, Tables 1-2")
print(f"  Notebook 02 → Feature matrix: 610 individuals × 1219 features")
print(f"  Notebook 03 → ML training: XGBoost AUC=0.9957, Table 3, Figures 4-5")
print(f"  Notebook 04 → SHAP: 15/20 features population-specific, Figures 6-8")
print(f"  Notebook 05 → CPIC mapping: clinical dosing alerts, Figures 9-10")
print(f"\nNext steps:")
print(f"  1. Commit notebook 05 and results")
print(f"  2. Write manuscript sections using paper sentences from each notebook")
print(f"  3. Draft professor outreach emails with GitHub link + key results")
print(f"  4. Apply GKS + MEXT scholarships in November")
